In [ ]:
!pip install -q google-generativeai pandas numpy sentence-transformers faiss-cpu scikit-learn rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 24.0 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets pandas pyarrow

from datasets import load_dataset
import pandas as pd

# Start with a small, fast category. Switch later if we want more variety.
CATEGORY = "All_Beauty"

reviews_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"raw_review_{CATEGORY}",
    split="full"
)
meta_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"raw_meta_{CATEGORY}",
    split="full"
)

reviews = reviews_ds.to_pandas()
meta    = meta_ds.to_pandas()

print('reviews:', reviews.shape, '| cols:', list(reviews.columns))
print('meta   :', meta.shape,    '| cols:', list(meta.columns))
print('\nfirst review:')
print(reviews.iloc[0].to_dict())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


raw/review_categories/All_Beauty.jsonl:   0%|          | 0.00/327M [00:00<?, ?B/s]

Generating full split: 0 examples [00:00, ? examples/s]

raw/meta_categories/meta_All_Beauty.json(…):   0%|          | 0.00/213M [00:00<?, ?B/s]

Generating full split:   0%|          | 0/112590 [00:00<?, ? examples/s]

reviews: (701528, 10) | cols: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
meta   : (112590, 16) | cols: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

first review:
{'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': array([], dtype=object), 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': True}


In [ ]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
model = genai.GenerativeModel('gemini-2.5-flash')

resp = model.generate_content(
    "You are a 24-year-old software engineer from Lagos. "
    "In 2 sentences, write a Yelp-style review of a jollof rice spot in Lekki. "
    "Use natural Nigerian English. Don't overdo pidgin."
)
print(resp.text)

Their jollof rice here is seriously good, giving you that proper party jollof feel with the smoky flavour on point. Honestly, if you're in Lekki and craving the real deal, this is the spot you need to check out.


In [ ]:
!pip install -q "datasets<4.0"
print("Now click Runtime → Restart session, then re-run Cell 1, 2, 3, 4 in order.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 26.0 MB/s eta 0:00:00
Now click Runtime → Restart session, then re-run Cell 1, 2, 3, 4 in order.


In [ ]:
# Keep users with enough history (>=5 reviews)
counts = reviews.groupby('user_id').size()
power_users = counts[counts >= 5].index
reviews_pu = reviews[reviews['user_id'].isin(power_users)].copy()

# Attach item metadata
meta_slim = meta[['parent_asin', 'title', 'categories', 'store', 'description']].rename(
    columns={'title': 'item_title', 'categories': 'item_categories'}
)
reviews_pu = reviews_pu.merge(meta_slim, on='parent_asin', how='left')

# timestamps are ms since epoch
reviews_pu['date'] = pd.to_datetime(reviews_pu['timestamp'], unit='ms')

print(f'power users (>=5 reviews): {len(power_users):,}')
print(f'their reviews            : {len(reviews_pu):,}')
print(f'avg reviews / power user : {len(reviews_pu)/max(len(power_users),1):.1f}')
reviews_pu[['user_id', 'asin', 'rating', 'date', 'item_title']].head(3)

NameError: name 'reviews' is not defined

In [1]:
!pip install -q pandas pyarrow google-generativeai rouge-score bert-score tqdm scikit-learn httpx fastapi "uvicorn[standard]" sentence-transformers faiss-cpu

import pandas as pd, os, pickle, json

BASE = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
CATEGORY = "All_Beauty"

print("Downloading reviews JSONL...")
reviews = pd.read_json(f"{BASE}/review_categories/{CATEGORY}.jsonl", lines=True)
print(f"reviews: {reviews.shape}")

print("Downloading meta JSONL...")
meta = pd.read_json(f"{BASE}/meta_categories/meta_{CATEGORY}.jsonl", lines=True)
print(f"meta: {meta.shape}")

counts = reviews.groupby('user_id').size()
power_users = counts[counts >= 5].index
reviews_pu = reviews[reviews['user_id'].isin(power_users)].copy()
meta_slim = meta[['parent_asin','title','categories','store','description']].rename(
    columns={'title':'item_title','categories':'item_categories'})
reviews_pu = reviews_pu.merge(meta_slim, on='parent_asin', how='left')
reviews_pu['date'] = pd.to_datetime(reviews_pu['timestamp'], unit='ms')

reviews_pu = reviews_pu.sort_values(['user_id','date'])
reviews_pu['rank'] = reviews_pu.groupby('user_id').cumcount(ascending=False)
test_all  = reviews_pu[reviews_pu['rank'] == 0]
train_all = reviews_pu[reviews_pu['rank'] >  0]
SAMPLE_USERS = test_all['user_id'].sample(n=min(500, len(test_all)), random_state=42).tolist()
test  = test_all[test_all['user_id'].isin(SAMPLE_USERS)].copy()
train = train_all[train_all['user_id'].isin(SAMPLE_USERS)].copy()

def build_persona_pack(user_rows, max_history=10):
    rows = user_rows.sort_values('date', ascending=False).head(max_history)
    history = [{
        'item'   : r['item_title'] if pd.notna(r['item_title']) else 'unknown',
        'rating' : int(r['rating']),
        'excerpt': (r['text'] or '')[:200],
        'date'   : r['date'].strftime('%Y-%m'),
    } for _, r in rows.iterrows()]
    return {
        'user_id'             : rows['user_id'].iloc[0],
        'avg_rating'          : round(rows['rating'].mean(), 2),
        'rating_distribution' : rows['rating'].value_counts(normalize=True).round(2).to_dict(),
        'avg_review_length'   : int(rows['text'].str.len().mean()),
        'history_count'       : len(rows),
        'history'             : history,
    }

personas = {}
for uid in SAMPLE_USERS:
    ut = train[train['user_id'] == uid]
    if len(ut) >= 4:
        personas[uid] = build_persona_pack(ut)

os.makedirs('/content/work', exist_ok=True)
with open('/content/work/personas.pkl', 'wb') as f:
    pickle.dump(personas, f)
test.to_parquet('/content/work/test_heldout.parquet', index=False)
train.to_parquet('/content/work/train_history.parquet', index=False)
print(f'\nready: {len(personas)} personas, {len(test)} held-out items')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.4 MB/s eta 0:00:00
reviews: (701528, 10)
meta: (112590, 14)

ready: 500 personas, 500 held-out items


In [2]:
from google.colab import userdata
import google.generativeai as genai, os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ['GEMINI_API_KEY'])

APP_DIR = '/content/app'
os.makedirs(APP_DIR, exist_ok=True)

# requirements.txt
with open(f'{APP_DIR}/requirements.txt', 'w') as f:
    f.write("fastapi==0.115.0\nuvicorn[standard]==0.32.0\ngoogle-generativeai==0.8.3\npydantic==2.9.2\nsentence-transformers==3.0.1\nfaiss-cpu==1.8.0\npandas==2.2.2\nnumpy==1.26.4\n")

# Dockerfile
with open(f'{APP_DIR}/Dockerfile', 'w') as f:
    f.write("FROM python:3.11-slim\nWORKDIR /app\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY . .\nENV PORT=7860\nEXPOSE 7860\nCMD [\"sh\",\"-c\",\"uvicorn app:app --host 0.0.0.0 --port ${PORT}\"]\n")

print('skeleton written:', os.listdir(APP_DIR))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


skeleton written: ['Dockerfile', 'requirements.txt']


In [3]:
%%writefile /content/app/agent.py
import os, json, re, time
import google.generativeai as genai

def format_persona_for_llm(persona: dict) -> str:
    lines = [
        "User behavioural profile:",
        f"- Average rating given : {persona.get('avg_rating','n/a')}/5",
        f"- Rating distribution  : {persona.get('rating_distribution',{})}",
        f"- Typical review length: ~{persona.get('avg_review_length','n/a')} chars",
        f"- Reviews on record    : {persona.get('history_count', len(persona.get('history',[])))}",
        "",
        "Recent reviews (newest first):",
    ]
    for h in persona.get('history', [])[:8]:
        lines.append(f'  [{h.get("date","")}] "{(h.get("item","unknown"))[:80]}" - {h.get("rating",0)} stars')
        if h.get('excerpt'):
            lines.append(f'    excerpt: {h["excerpt"]}')
    return "\n".join(lines)


class ReviewSimulatorAgent:
    def __init__(self, model_name='gemini-2.5-flash', naija_mode=False, api_key=None):
        api_key = api_key or os.environ.get('GEMINI_API_KEY')
        if not api_key: raise ValueError('GEMINI_API_KEY not set')
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_name)
        self.naija_mode = naija_mode

    def _call(self, prompt, retries=4):
        for i in range(retries):
            try: return self.model.generate_content(prompt).text.strip()
            except Exception as e:
                if '429' in str(e) or 'quota' in str(e).lower():
                    time.sleep(35 + i*10)
                elif i == retries - 1: raise
                else: time.sleep(2**i)

    def _extract_json(self, text):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.MULTILINE)
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m: text = m.group(0)
        return json.loads(text)

    def _naija_overlay(self):
        if not self.naija_mode: return ""
        return ("\nIMPORTANT: This user is Nigerian. Use natural Nigerian English with "
                "comfortable code-switching and light pidgin where it fits. Don't caricature.\n")

    def simulate(self, persona, target_item):
        persona_text = format_persona_for_llm(persona)
        item_text = (
            f"Item title : {target_item.get('title','unknown')}\n"
            f"Categories : {target_item.get('categories', [])}\n"
            f"Description: {(target_item.get('description') or '')[:300]}"
        )
        analysis = self._call(
            f"{persona_text}\n\nIn 2-3 sentences, describe this user's reviewing personality."
        )
        gen_prompt = (
            f"{persona_text}\n\nPersonality analysis:\n{analysis}\n\n"
            f"They are reviewing a NEW item:\n{item_text}\n{self._naija_overlay()}\n"
            "Predict (a) the integer star rating (1-5) and (b) the review THEY would write.\n\n"
            "Respond with ONLY this JSON:\n"
            '{"predicted_rating": <int>, "review_text": "<the review>", "reasoning": "<1 sentence>"}'
        )
        raw = self._call(gen_prompt)
        try:
            out = self._extract_json(raw)
            out['predicted_rating'] = int(round(float(out['predicted_rating'])))
            out['analysis'] = analysis
            return out
        except Exception as e:
            return {'predicted_rating': 3, 'review_text': raw[:500],
                    'reasoning': f'parse_error: {e}', 'analysis': analysis}

Writing /content/app/agent.py


In [4]:
%%writefile /content/app/recommender.py
import os, json, re, time, pickle
import numpy as np
import pandas as pd
import faiss
import google.generativeai as genai
from sentence_transformers import SentenceTransformer


class RecommendationAgent:
    def __init__(self, catalog, index, embedder, model_name='gemini-2.5-flash', api_key=None):
        api_key = api_key or os.environ.get('GEMINI_API_KEY')
        if not api_key: raise ValueError('GEMINI_API_KEY not set')
        genai.configure(api_key=api_key)
        self.catalog = catalog
        self.index = index
        self.embedder = embedder
        self.model = genai.GenerativeModel(model_name)

    def _call(self, prompt, retries=4):
        for i in range(retries):
            try: return self.model.generate_content(prompt).text.strip()
            except Exception as e:
                if '429' in str(e) or 'quota' in str(e).lower():
                    time.sleep(35 + i*10)
                elif i == retries - 1: raise
                else: time.sleep(2**i)

    def _extract_json(self, text):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.MULTILINE)
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m: text = m.group(0)
        return json.loads(text)

    def _build_query(self, persona, conversation_context=None):
        history_text = '\n'.join(
            f"- {h['rating']}* '{h['item'][:60]}': {h['excerpt'][:100]}"
            for h in persona.get('history', [])[:8]
        )
        cold_start = len(persona.get('history', [])) == 0
        prompt = (
            (f"User review history:\n{history_text}\n" if not cold_start else "User has no prior history (COLD START).\n")
            + f"Average rating: {persona.get('avg_rating','unknown')}\n"
            + (f"Conversation context: {conversation_context}\n" if conversation_context else "")
            + "\nIn one sentence, describe what kind of product this user is most likely to want next. "
            "Be SPECIFIC about product types, key attributes, and qualities they care about."
        )
        return self._call(prompt)

    def _retrieve(self, query_text, exclude_asins=None, k=200):
        qvec = self.embedder.encode([query_text]).astype('float32')
        faiss.normalize_L2(qvec)
        scores, idx = self.index.search(qvec, k)
        cands = self.catalog.iloc[idx[0]].copy()
        cands['retrieval_score'] = scores[0]
        if exclude_asins:
            cands = cands[~cands['parent_asin'].isin(exclude_asins)]
        return cands

    def _rerank(self, persona, candidates, top_k=10, feed_top=40):
        cand_text = '\n'.join(
            f"{i+1}. [{row['parent_asin']}] {row['title'][:80]} (avg {row['average_rating']}*)"
            for i, (_, row) in enumerate(candidates.head(feed_top).iterrows())
        )
        history_text = '\n'.join(
            f"- {h['rating']}* '{h['item'][:50]}'"
            for h in persona.get('history', [])[:6]
        )
        prompt = (
            f"User profile (avg rating {persona.get('avg_rating','n/a')}):\n{history_text}\n\n"
            f"Candidate items (use these EXACT parent_asin codes):\n{cand_text}\n\n"
            f"Pick the top {top_k} items most likely to delight this user. "
            "Respond with ONLY this JSON:\n"
            '{"recommendations":[{"parent_asin":"<exact code>","rank":1,"reason":"<1 sentence>"}]}'
        )
        raw = self._call(prompt)
        try:
            recs = self._extract_json(raw).get('recommendations', [])
            valid = set(candidates.head(feed_top)['parent_asin'])
            recs = [r for r in recs if r.get('parent_asin') in valid]
            if len(recs) < top_k:
                seen = {r['parent_asin'] for r in recs}
                for _, row in candidates.head(feed_top).iterrows():
                    if row['parent_asin'] not in seen:
                        recs.append({'parent_asin': row['parent_asin'], 'rank': len(recs)+1,
                                     'reason': 'retrieval_fallback'})
                    if len(recs) >= top_k: break
            return recs[:top_k]
        except Exception:
            return [{'parent_asin': row['parent_asin'], 'rank': i+1, 'reason': 'parse_fallback'}
                    for i, (_, row) in enumerate(candidates.head(top_k).iterrows())]

    def recommend(self, persona, exclude_asins=None, conversation_context=None, top_k=10):
        query = self._build_query(persona, conversation_context)
        cands = self._retrieve(query, exclude_asins, k=200)
        recs = self._rerank(persona, cands, top_k=top_k, feed_top=40)
        for r in recs:
            row = self.catalog[self.catalog['parent_asin'] == r['parent_asin']]
            if not row.empty:
                r['title'] = row.iloc[0]['title']
                r['categories'] = row.iloc[0]['categories'] if isinstance(row.iloc[0]['categories'], list) else []
        return {'query': query, 'recommendations': recs}

Writing /content/app/recommender.py


In [5]:
import numpy as np, faiss, pickle
from sentence_transformers import SentenceTransformer

catalog = meta.copy()
catalog = catalog[catalog['title'].notna() & (catalog['title'].str.len() > 5)].copy()
catalog = catalog[['parent_asin','title','categories','description','average_rating','rating_number','store']]

def cats_to_str(x): return ', '.join(x) if isinstance(x, list) else str(x or '')
def desc_to_str(x):
    if isinstance(x, list): return ' '.join(str(s) for s in x[:2])
    return str(x or '')

catalog['text_repr'] = (catalog['title'].fillna('') + ' | ' +
                       catalog['categories'].apply(cats_to_str) + ' | ' +
                       catalog['description'].apply(desc_to_str)).str.slice(0, 300)

catalog = catalog.head(5000).reset_index(drop=True)
print(f'catalog size: {len(catalog)}')

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(catalog['text_repr'].tolist(), show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, '/content/work/catalog.faiss')
catalog.to_parquet('/content/work/catalog.parquet', index=False)
print(f'FAISS index: {index.ntotal} vectors')

catalog size: 5000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

FAISS index: 5000 vectors


In [6]:
import shutil, os

# Make catalog + FAISS available to the container
os.makedirs('/content/app/data', exist_ok=True)
shutil.copy('/content/work/catalog.parquet', '/content/app/data/catalog.parquet')
shutil.copy('/content/work/catalog.faiss',   '/content/app/data/catalog.faiss')
print('catalog files copied to app dir')

catalog files copied to app dir


In [7]:
%%writefile /content/app/app.py
import os, pickle
from typing import List, Optional, Dict, Any
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer

from agent import ReviewSimulatorAgent
from recommender import RecommendationAgent

# ---------- Startup ----------
print("Loading catalog and FAISS index...")
CATALOG  = pd.read_parquet(os.path.join(os.path.dirname(__file__), 'data', 'catalog.parquet'))
FAISS_IX = faiss.read_index(os.path.join(os.path.dirname(__file__), 'data', 'catalog.faiss'))
EMBEDDER = SentenceTransformer('all-MiniLM-L6-v2')
REC_AGENT = RecommendationAgent(CATALOG, FAISS_IX, EMBEDDER)
print(f"ready: {len(CATALOG)} items, {FAISS_IX.ntotal} embeddings")


# ---------- Schemas ----------
class HistoryItem(BaseModel):
    item: str
    rating: int
    excerpt: Optional[str] = ""
    date: Optional[str] = ""

class Persona(BaseModel):
    user_id: Optional[str] = None
    avg_rating: float = Field(3.5, ge=1.0, le=5.0)
    rating_distribution: Dict[str, float] = {}
    avg_review_length: int = 200
    history_count: Optional[int] = None
    history: List[HistoryItem] = []

class TargetItem(BaseModel):
    title: str
    categories: List[str] = []
    description: Optional[str] = ""

# Task A
class SimulateRequest(BaseModel):
    persona: Persona
    target_item: TargetItem
    naija_mode: bool = False

class SimulateResponse(BaseModel):
    predicted_rating: int
    review_text: str
    reasoning: str
    analysis: Optional[str] = None

# Task B
class RecommendRequest(BaseModel):
    persona: Persona
    exclude_asins: List[str] = []
    conversation_context: Optional[str] = None
    top_k: int = 10

class Recommendation(BaseModel):
    parent_asin: str
    rank: int
    reason: str
    title: Optional[str] = None
    categories: Optional[List[str]] = None

class RecommendResponse(BaseModel):
    query: str
    recommendations: List[Recommendation]


# ---------- App ----------
app = FastAPI(title="DSN x BCT - Combined Agent Service", version="1.0.0")

@app.get("/")
def root():
    return {"service": "dsn-bct-agents",
            "tasks": ["A: /simulate-review", "B: /recommend"],
            "status": "ok"}

@app.get("/health")
def health():
    return {"status": "ok",
            "catalog_size": len(CATALOG),
            "faiss_vectors": FAISS_IX.ntotal}

@app.post("/simulate-review", response_model=SimulateResponse)
def simulate_review(req: SimulateRequest):
    try:
        agent = ReviewSimulatorAgent(naija_mode=req.naija_mode)
        return agent.simulate(req.persona.model_dump(), req.target_item.model_dump())
    except ValueError as e:
        raise HTTPException(500, str(e))
    except Exception as e:
        raise HTTPException(500, f"agent_error: {e}")

@app.post("/recommend", response_model=RecommendResponse)
def recommend(req: RecommendRequest):
    try:
        result = REC_AGENT.recommend(
            req.persona.model_dump(),
            exclude_asins=set(req.exclude_asins) if req.exclude_asins else None,
            conversation_context=req.conversation_context,
            top_k=req.top_k,
        )
        return result
    except ValueError as e:
        raise HTTPException(500, str(e))
    except Exception as e:
        raise HTTPException(500, f"agent_error: {e}")

Writing /content/app/app.py


In [8]:
import sys, os, json
sys.path.insert(0, '/content/app')

# clean module cache so the latest app.py is picked up
for m in ['app', 'agent', 'recommender']:
    sys.modules.pop(m, None)

from fastapi.testclient import TestClient
from app import app
client = TestClient(app)

print('health:', client.get('/health').json())

# Pick one persona we have in memory
sample = next(iter(personas.values()))
sample_clean = {
    'user_id'             : sample['user_id'],
    'avg_rating'          : sample['avg_rating'],
    'rating_distribution' : {str(k): v for k, v in sample['rating_distribution'].items()},
    'avg_review_length'   : sample['avg_review_length'],
    'history_count'       : sample['history_count'],
    'history'             : sample['history'],
}

# --- Task A ---
print('\n--- POST /simulate-review ---')
r = client.post('/simulate-review', json={
    "persona": sample_clean,
    "target_item": {
        "title": "Vitamin C Brightening Serum 30ml",
        "categories": ["Beauty", "Skin Care"],
        "description": "20% Vitamin C with hyaluronic acid. Brightens, hydrates, evens tone."
    },
    "naija_mode": False
}, timeout=120)
print('status:', r.status_code)
print(json.dumps(r.json(), indent=2)[:600])

# --- Task B ---
print('\n--- POST /recommend ---')
r = client.post('/recommend', json={
    "persona": sample_clean,
    "exclude_asins": [],
    "top_k": 5
}, timeout=180)
print('status:', r.status_code)
out = r.json()
print('query:', out.get('query'))
for rec in out.get('recommendations', [])[:5]:
    print(f"  #{rec['rank']} {rec.get('title','')[:60]} -- {rec.get('reason','')[:80]}")

Loading catalog and FAISS index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ready: 5000 items, 5000 embeddings
health: {'status': 'ok', 'catalog_size': 5000, 'faiss_vectors': 5000}

--- POST /simulate-review ---


status: 500
{
  "detail": "agent_error: 'NoneType' object is not subscriptable"
}

--- POST /recommend ---


status: 500
query: None


In [10]:
!pip install -q openai

from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('Groq key loaded:', bool(os.environ.get('GROQ_API_KEY')))

Groq key loaded: True


In [11]:
%%writefile /content/app/agent.py
import os, json, re, time
from openai import OpenAI

GROQ_BASE  = "https://api.groq.com/openai/v1"
GROQ_MODEL = "llama-3.3-70b-versatile"


def format_persona_for_llm(persona: dict) -> str:
    lines = [
        "User behavioural profile:",
        f"- Average rating given : {persona.get('avg_rating','n/a')}/5",
        f"- Rating distribution  : {persona.get('rating_distribution',{})}",
        f"- Typical review length: ~{persona.get('avg_review_length','n/a')} chars",
        f"- Reviews on record    : {persona.get('history_count', len(persona.get('history',[])))}",
        "",
        "Recent reviews (newest first):",
    ]
    for h in persona.get('history', [])[:8]:
        lines.append(f'  [{h.get("date","")}] "{(h.get("item","unknown"))[:80]}" - {h.get("rating",0)} stars')
        if h.get('excerpt'):
            lines.append(f'    excerpt: {h["excerpt"]}')
    return "\n".join(lines)


class ReviewSimulatorAgent:
    def __init__(self, model_name=GROQ_MODEL, naija_mode=False, api_key=None):
        api_key = api_key or os.environ.get('GROQ_API_KEY')
        if not api_key: raise ValueError('GROQ_API_KEY not set')
        self.client = OpenAI(api_key=api_key, base_url=GROQ_BASE)
        self.model_name = model_name
        self.naija_mode = naija_mode

    def _call(self, prompt, retries=4):
        for i in range(retries):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.7,
                )
                return resp.choices[0].message.content.strip()
            except Exception as e:
                msg = str(e)
                if '429' in msg or 'rate' in msg.lower():
                    time.sleep(15 + i*5)
                elif i == retries - 1:
                    raise
                else:
                    time.sleep(2 ** i)
        raise RuntimeError("LLM call failed after retries")

    def _extract_json(self, text):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.MULTILINE)
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m: text = m.group(0)
        return json.loads(text)

    def _naija_overlay(self):
        if not self.naija_mode: return ""
        return ("\nIMPORTANT: This user is Nigerian. Use natural Nigerian English with "
                "comfortable code-switching and light pidgin where it fits. Don't caricature.\n")

    def simulate(self, persona, target_item):
        persona_text = format_persona_for_llm(persona)
        item_text = (
            f"Item title : {target_item.get('title','unknown')}\n"
            f"Categories : {target_item.get('categories', [])}\n"
            f"Description: {(target_item.get('description') or '')[:300]}"
        )
        analysis = self._call(
            f"{persona_text}\n\nIn 2-3 sentences, describe this user's reviewing personality."
        )
        gen_prompt = (
            f"{persona_text}\n\nPersonality analysis:\n{analysis}\n\n"
            f"They are reviewing a NEW item:\n{item_text}\n{self._naija_overlay()}\n"
            "Predict (a) the integer star rating (1-5) and (b) the review THEY would write.\n\n"
            "Respond with ONLY this JSON:\n"
            '{"predicted_rating": <int>, "review_text": "<the review>", "reasoning": "<1 sentence>"}'
        )
        raw = self._call(gen_prompt)
        try:
            out = self._extract_json(raw)
            out['predicted_rating'] = int(round(float(out['predicted_rating'])))
            out['analysis'] = analysis
            return out
        except Exception as e:
            return {'predicted_rating': 3, 'review_text': raw[:500],
                    'reasoning': f'parse_error: {e}', 'analysis': analysis}

Overwriting /content/app/agent.py


In [12]:
%%writefile /content/app/recommender.py
import os, json, re, time
import numpy as np, pandas as pd, faiss
from openai import OpenAI
from sentence_transformers import SentenceTransformer

GROQ_BASE  = "https://api.groq.com/openai/v1"
GROQ_MODEL = "llama-3.3-70b-versatile"


class RecommendationAgent:
    def __init__(self, catalog, index, embedder, model_name=GROQ_MODEL, api_key=None):
        api_key = api_key or os.environ.get('GROQ_API_KEY')
        if not api_key: raise ValueError('GROQ_API_KEY not set')
        self.client = OpenAI(api_key=api_key, base_url=GROQ_BASE)
        self.model_name = model_name
        self.catalog = catalog
        self.index = index
        self.embedder = embedder

    def _call(self, prompt, retries=4):
        for i in range(retries):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.5,
                )
                return resp.choices[0].message.content.strip()
            except Exception as e:
                msg = str(e)
                if '429' in msg or 'rate' in msg.lower():
                    time.sleep(15 + i*5)
                elif i == retries - 1:
                    raise
                else:
                    time.sleep(2 ** i)
        raise RuntimeError("LLM call failed after retries")

    def _extract_json(self, text):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.MULTILINE)
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m: text = m.group(0)
        return json.loads(text)

    def _build_query(self, persona, conversation_context=None):
        history_text = '\n'.join(
            f"- {h['rating']}* '{h['item'][:60]}': {h['excerpt'][:100]}"
            for h in persona.get('history', [])[:8]
        )
        cold_start = len(persona.get('history', [])) == 0
        prompt = (
            (f"User review history:\n{history_text}\n" if not cold_start else "User has no prior history (COLD START).\n")
            + f"Average rating: {persona.get('avg_rating','unknown')}\n"
            + (f"Conversation context: {conversation_context}\n" if conversation_context else "")
            + "\nIn one sentence, describe what kind of product this user is most likely to want next. "
            "Be SPECIFIC about product types, key attributes, and qualities they care about."
        )
        return self._call(prompt)

    def _retrieve(self, query_text, exclude_asins=None, k=200):
        qvec = self.embedder.encode([query_text]).astype('float32')
        faiss.normalize_L2(qvec)
        scores, idx = self.index.search(qvec, k)
        cands = self.catalog.iloc[idx[0]].copy()
        cands['retrieval_score'] = scores[0]
        if exclude_asins:
            cands = cands[~cands['parent_asin'].isin(exclude_asins)]
        return cands

    def _rerank(self, persona, candidates, top_k=10, feed_top=40):
        cand_text = '\n'.join(
            f"{i+1}. [{row['parent_asin']}] {row['title'][:80]} (avg {row['average_rating']}*)"
            for i, (_, row) in enumerate(candidates.head(feed_top).iterrows())
        )
        history_text = '\n'.join(
            f"- {h['rating']}* '{h['item'][:50]}'"
            for h in persona.get('history', [])[:6]
        )
        prompt = (
            f"User profile (avg rating {persona.get('avg_rating','n/a')}):\n{history_text}\n\n"
            f"Candidate items (use these EXACT parent_asin codes):\n{cand_text}\n\n"
            f"Pick the top {top_k} items most likely to delight this user. "
            "Respond with ONLY this JSON:\n"
            '{"recommendations":[{"parent_asin":"<exact code>","rank":1,"reason":"<1 sentence>"}]}'
        )
        raw = self._call(prompt)
        try:
            recs = self._extract_json(raw).get('recommendations', [])
            valid = set(candidates.head(feed_top)['parent_asin'])
            recs = [r for r in recs if r.get('parent_asin') in valid]
            if len(recs) < top_k:
                seen = {r['parent_asin'] for r in recs}
                for _, row in candidates.head(feed_top).iterrows():
                    if row['parent_asin'] not in seen:
                        recs.append({'parent_asin': row['parent_asin'], 'rank': len(recs)+1,
                                     'reason': 'retrieval_fallback'})
                    if len(recs) >= top_k: break
            return recs[:top_k]
        except Exception:
            return [{'parent_asin': row['parent_asin'], 'rank': i+1, 'reason': 'parse_fallback'}
                    for i, (_, row) in enumerate(candidates.head(top_k).iterrows())]

    def recommend(self, persona, exclude_asins=None, conversation_context=None, top_k=10):
        query = self._build_query(persona, conversation_context)
        cands = self._retrieve(query, exclude_asins, k=200)
        recs = self._rerank(persona, cands, top_k=top_k, feed_top=40)
        for r in recs:
            row = self.catalog[self.catalog['parent_asin'] == r['parent_asin']]
            if not row.empty:
                r['title'] = row.iloc[0]['title']
                r['categories'] = row.iloc[0]['categories'] if isinstance(row.iloc[0]['categories'], list) else []
        return {'query': query, 'recommendations': recs}

Overwriting /content/app/recommender.py


In [13]:
# Update requirements (the container will need openai instead of google-generativeai)
with open('/content/app/requirements.txt', 'w') as f:
    f.write("fastapi==0.115.0\nuvicorn[standard]==0.32.0\nopenai==1.51.0\npydantic==2.9.2\nsentence-transformers==3.0.1\nfaiss-cpu==1.8.0\npandas==2.2.2\nnumpy==1.26.4\n")

import sys, os, json
sys.path.insert(0, '/content/app')
for m in ['app', 'agent', 'recommender']:
    sys.modules.pop(m, None)

from fastapi.testclient import TestClient
from app import app
client = TestClient(app)

print('health:', client.get('/health').json())

sample = next(iter(personas.values()))
sample_clean = {
    'user_id'             : sample['user_id'],
    'avg_rating'          : sample['avg_rating'],
    'rating_distribution' : {str(k): v for k, v in sample['rating_distribution'].items()},
    'avg_review_length'   : sample['avg_review_length'],
    'history_count'       : sample['history_count'],
    'history'             : sample['history'],
}

print('\n--- POST /simulate-review ---')
r = client.post('/simulate-review', json={
    "persona": sample_clean,
    "target_item": {
        "title": "Vitamin C Brightening Serum 30ml",
        "categories": ["Beauty", "Skin Care"],
        "description": "20% Vitamin C with hyaluronic acid. Brightens, hydrates, evens tone."
    },
    "naija_mode": False
}, timeout=120)
print('status:', r.status_code)
print(json.dumps(r.json(), indent=2)[:500])

print('\n--- POST /recommend ---')
r = client.post('/recommend', json={"persona": sample_clean, "exclude_asins": [], "top_k": 5}, timeout=180)
print('status:', r.status_code)
out = r.json()
print('query:', out.get('query'))
for rec in (out.get('recommendations') or [])[:5]:
    print(f"  #{rec['rank']} {rec.get('title','')[:60]} -- {rec.get('reason','')[:80]}")

Loading catalog and FAISS index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ready: 5000 items, 5000 embeddings
health: {'status': 'ok', 'catalog_size': 5000, 'faiss_vectors': 5000}

--- POST /simulate-review ---
status: 200
{
  "predicted_rating": 5,
  "review_text": "I'm absolutely loving this Vitamin C Brightening Serum! It's so easy to apply and absorbs quickly into my skin, leaving it feeling hydrated and looking brighter. The combination of 20% Vitamin C and hyaluronic acid is a game changer - my skin tone is more even and I've noticed a reduction in fine lines. I've been using it for a few weeks now and I'm so impressed with the results. Definitely a new staple in my skincare routine!",
  "reasoning": "The us

--- POST /recommend ---
status: 200
query: This user is most likely to want a skincare product, specifically a moisturizing or hydrating treatment with natural ingredients, that is easy to use, effective, and possibly has exfoliating or regenerative properties, with a preference for products that are simple, gentle, and suitable for dry or sensitiv

In [14]:
import time, json, numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer
from sklearn.metrics import mean_squared_error

sys.path.insert(0, '/content/app')
from agent import ReviewSimulatorAgent

N_EVAL_A = 15
eligible = test[test['user_id'].isin(personas)].head(N_EVAL_A)

agent_a = ReviewSimulatorAgent(naija_mode=False)
item_lookup = meta.set_index('parent_asin').to_dict('index')

def get_target_item(parent_asin):
    m = item_lookup.get(parent_asin, {})
    return {
        'title'      : m.get('title', 'Unknown'),
        'categories' : m.get('categories', []),
        'description': str(m.get('description', ''))[:500],
    }

results_a = []
for _, row in tqdm(eligible.iterrows(), total=len(eligible)):
    uid = row['user_id']
    target = get_target_item(row['parent_asin'])
    try:
        pred = agent_a.simulate(personas[uid], target)
        results_a.append({
            'user_id'         : uid,
            'item'            : target['title'],
            'actual_rating'   : int(row['rating']),
            'predicted_rating': pred['predicted_rating'],
            'actual_text'     : row['text'],
            'predicted_text'  : pred['review_text'],
            'reasoning'       : pred['reasoning'],
        })
    except Exception as e:
        results_a.append({'user_id': uid, 'error': str(e)})
    time.sleep(2)  # respect Groq TPM

res_a_df = pd.DataFrame(results_a).dropna(subset=['predicted_rating'])

rmse  = np.sqrt(mean_squared_error(res_a_df['actual_rating'], res_a_df['predicted_rating']))
mae   = np.mean(np.abs(res_a_df['actual_rating'] - res_a_df['predicted_rating']))
exact = (res_a_df['actual_rating'] == res_a_df['predicted_rating']).mean()
within_1 = (np.abs(res_a_df['actual_rating'] - res_a_df['predicted_rating']) <= 1).mean()

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [rouge.score(a, p)['rougeL'].fmeasure
                for a, p in zip(res_a_df['actual_text'], res_a_df['predicted_text'])]

metrics_a = {
    'n'           : len(res_a_df),
    'rmse'        : float(rmse),
    'mae'         : float(mae),
    'exact_match' : float(exact),
    'within_1_star': float(within_1),
    'rouge_l_avg' : float(np.mean(rouge_scores)),
}

print('\n=== Task A metrics ===')
for k, v in metrics_a.items():
    print(f'  {k:<14}: {v:.3f}' if isinstance(v, float) else f'  {k:<14}: {v}')

res_a_df['rouge_l'] = rouge_scores
res_a_df.to_parquet('/content/work/results_task_a.parquet', index=False)
with open('/content/work/metrics_task_a.json', 'w') as f:
    json.dump(metrics_a, f, indent=2)
print('saved -> /content/work/')

100%|██████████| 15/15 [00:45<00:00,  3.03s/it]


=== Task A metrics ===
  n             : 15
  rmse          : 1.366
  mae           : 0.667
  exact_match   : 0.667
  within_1_star : 0.867
  rouge_l_avg   : 0.256
saved -> /content/work/


In [15]:
from app import REC_AGENT  # already loaded

N_EVAL_B = 15
catalog_asins = set(CATALOG['parent_asin'])
eval_pool = test[test['parent_asin'].isin(catalog_asins) & test['user_id'].isin(personas)].head(N_EVAL_B)
print(f'eval pool: {len(eval_pool)} users')

results_b = []
for _, row in tqdm(eval_pool.iterrows(), total=len(eval_pool)):
    uid = row['user_id']
    target_asin = row['parent_asin']
    user_history_asins = set(train[train['user_id'] == uid]['parent_asin'])
    try:
        rec_result = REC_AGENT.recommend(personas[uid], exclude_asins=user_history_asins, top_k=10)
        rec_asins = [r['parent_asin'] for r in rec_result['recommendations']]
        if target_asin in rec_asins:
            rank = rec_asins.index(target_asin) + 1
            hit, ndcg = True, 1.0 / np.log2(rank + 1)
        else:
            rank, hit, ndcg = None, False, 0.0
        results_b.append({'user_id': uid, 'target_asin': target_asin, 'rank': rank,
                          'hit_at_10': hit, 'ndcg_at_10': ndcg, 'top_recs': rec_asins})
    except Exception as e:
        results_b.append({'user_id': uid, 'error': str(e)})
    time.sleep(3)

res_b_df = pd.DataFrame(results_b)
hits = res_b_df['hit_at_10'].dropna().mean()
ndcg = res_b_df['ndcg_at_10'].dropna().mean()

metrics_b = {
    'n'         : int(res_b_df['hit_at_10'].notna().sum()),
    'hit_at_10' : float(hits),
    'ndcg_at_10': float(ndcg),
}
print('\n=== Task B metrics ===')
for k, v in metrics_b.items():
    print(f'  {k:<12}: {v:.3f}' if isinstance(v, float) else f'  {k:<12}: {v}')

res_b_df.to_parquet('/content/work/results_task_b.parquet', index=False)
with open('/content/work/metrics_task_b.json', 'w') as f:
    json.dump(metrics_b, f, indent=2)
print('saved -> /content/work/')

NameError: name 'CATALOG' is not defined

In [16]:
from app import REC_AGENT, CATALOG

In [17]:
import time, json
import numpy as np
import pandas as pd
from tqdm import tqdm

from app import REC_AGENT, CATALOG

N_EVAL_B = 15
catalog_asins = set(CATALOG['parent_asin'])
eval_pool = test[test['parent_asin'].isin(catalog_asins) & test['user_id'].isin(personas)].head(N_EVAL_B)
print(f'eval pool: {len(eval_pool)} users')

results_b = []
for _, row in tqdm(eval_pool.iterrows(), total=len(eval_pool)):
    uid = row['user_id']
    target_asin = row['parent_asin']
    user_history_asins = set(train[train['user_id'] == uid]['parent_asin'])
    try:
        rec_result = REC_AGENT.recommend(personas[uid], exclude_asins=user_history_asins, top_k=10)
        rec_asins = [r['parent_asin'] for r in rec_result['recommendations']]
        if target_asin in rec_asins:
            rank = rec_asins.index(target_asin) + 1
            hit, ndcg = True, 1.0 / np.log2(rank + 1)
        else:
            rank, hit, ndcg = None, False, 0.0
        results_b.append({'user_id': uid, 'target_asin': target_asin, 'rank': rank,
                          'hit_at_10': hit, 'ndcg_at_10': ndcg, 'top_recs': rec_asins})
    except Exception as e:
        results_b.append({'user_id': uid, 'error': str(e)})
    time.sleep(3)

res_b_df = pd.DataFrame(results_b)
hits = res_b_df['hit_at_10'].dropna().mean()
ndcg = res_b_df['ndcg_at_10'].dropna().mean()

metrics_b = {
    'n'         : int(res_b_df['hit_at_10'].notna().sum()),
    'hit_at_10' : float(hits),
    'ndcg_at_10': float(ndcg),
}
print('\n=== Task B metrics ===')
for k, v in metrics_b.items():
    print(f'  {k:<12}: {v:.3f}' if isinstance(v, float) else f'  {k:<12}: {v}')

res_b_df.to_parquet('/content/work/results_task_b.parquet', index=False)
with open('/content/work/metrics_task_b.json', 'w') as f:
    json.dump(metrics_b, f, indent=2)
print('saved -> /content/work/')

eval pool: 15 users


100%|██████████| 15/15 [02:20<00:00,  9.36s/it]


=== Task B metrics ===
  n           : 15
  hit_at_10   : 0.067
  ndcg_at_10  : 0.022
saved -> /content/work/


In [18]:
import time, json, sys
sys.path.insert(0, '/content/app')
from agent import ReviewSimulatorAgent

p1 = {"user_id":"NG_chiamaka","avg_rating":3.8,
      "rating_distribution":{"5":0.3,"4":0.4,"3":0.2,"2":0.1},
      "avg_review_length":200,"history_count":4,
      "history":[
        {"item":"Hair oil from Aliexpress","rating":2,"date":"2024-09","excerpt":"6 weeks to arrive abeg. No growth after 2 months."},
        {"item":"Shea butter cream locally made","rating":5,"date":"2024-07","excerpt":"This one shock me. Skin dey shine well well."},
        {"item":"Detangling comb wide tooth","rating":4,"date":"2024-05","excerpt":"Solid build, does not snap my 4C hair."},
        {"item":"Lip balm tinted","rating":4,"date":"2024-03","excerpt":"Decent moisture. Pink reads more red on my skin."},
      ]}

p2 = {"user_id":"NG_tunde","avg_rating":4.2,
      "rating_distribution":{"5":0.5,"4":0.3,"3":0.15,"2":0.05},
      "avg_review_length":140,"history_count":4,
      "history":[
        {"item":"Beard trimmer cordless","rating":5,"date":"2024-10","excerpt":"Battery lasts well. No be small thing."},
        {"item":"Face wash oily skin","rating":4,"date":"2024-08","excerpt":"Works fine for harmattan. Price small high sha."},
        {"item":"Cologne fragrance dupe","rating":3,"date":"2024-06","excerpt":"Decent for the money. Sillage no too strong."},
        {"item":"Hair pomade strong hold","rating":5,"date":"2024-04","excerpt":"Holds my fade all day, even in Lagos heat."},
      ]}

p3 = {"user_id":"NG_zainab","avg_rating":2.9,
      "rating_distribution":{"5":0.1,"4":0.2,"3":0.3,"2":0.3,"1":0.1},
      "avg_review_length":280,"history_count":4,
      "history":[
        {"item":"Hijab pins set","rating":2,"date":"2024-11","excerpt":"Half were bent inside the pack. Quality control is zero."},
        {"item":"Black soap from Ghana","rating":4,"date":"2024-09","excerpt":"Authentic. Stripped my acne after 3 weeks."},
        {"item":"Lipstick matte finish","rating":1,"date":"2024-07","excerpt":"Dries my lips terribly. Shade nothing like picture."},
        {"item":"Body lotion shea blend","rating":3,"date":"2024-05","excerpt":"Average. Greasy then settles."},
      ]}

t1 = {"title":"Shea Butter Body Cream 500g handmade in Kano","categories":["Beauty","Body Care"],"description":"100 percent unrefined shea butter blended with coconut and lavender oil."}
t2 = {"title":"Generator-friendly clipper battery and AC","categories":["Grooming","Clippers"],"description":"Works on both NEPA and rechargeable battery. 90 min runtime. Ceramic blades."}
t3 = {"title":"Sunscreen SPF 50 for melanin-rich skin no white cast","categories":["Beauty","Sun Protection"],"description":"Lightweight tinted SPF 50 for darker skin tones. No ghost cast, non-greasy."}

agent_n = ReviewSimulatorAgent(naija_mode=True)
naija_results = []
for persona, target in zip([p1, p2, p3], [t1, t2, t3]):
    out = agent_n.simulate(persona, target)
    naija_results.append({"persona_id":persona["user_id"],"item":target["title"],"predicted_rating":out["predicted_rating"],"review_text":out["review_text"],"reasoning":out["reasoning"]})
    print("\n--- " + persona["user_id"] + " -> " + target["title"][:50] + " ---")
    print(str(out["predicted_rating"]) + " stars: " + out["review_text"])
    time.sleep(2)

with open("/content/work/naija_showcase.json", "w") as f:
    json.dump(naija_results, f, indent=2)
print("\nsaved -> /content/work/naija_showcase.json")


--- NG_chiamaka -> Shea Butter Body Cream 500g handmade in Kano ---
5 stars: This shea butter body cream na bomb, my skin dey glow like say I dey use glow up cream, the coconut and lavender oil blend well well, I love am

--- NG_tunde -> Generator-friendly clipper battery and AC ---
5 stars: Clips well, battery life dey, works with NEPA and generator, sharp blades too.

--- NG_zainab -> Sunscreen SPF 50 for melanin-rich skin no white ca ---
4 stars: This sunscreen dey try, no white cast at all and it's not greasy like some others I've used. Works well for my skin tone, I'm impressed

saved -> /content/work/naija_showcase.json


In [19]:
import time, json
from app import REC_AGENT

# === Cold-start: brand-new user with zero history ===
cold = {"user_id":"cold_user","avg_rating":3.5,"rating_distribution":{},"avg_review_length":0,"history_count":0,"history":[]}

print("=== COLD-START DEMO (no prior reviews) ===")
result_cold = REC_AGENT.recommend(cold, top_k=5)
print("Inferred intent:", result_cold['query'])
for r in result_cold['recommendations']:
    print(f"  #{r['rank']} {r.get('title','')[:60]}")
    print(f"     why: {r.get('reason','')[:100]}")

time.sleep(3)

# === Multi-turn: same user, two consecutive turns with refinement ===
existing = next(iter(personas.values()))
print("\n=== MULTI-TURN DEMO ===")
print(f"User: {existing['user_id']}")

print("\nTurn 1: no conversation context")
result_t1 = REC_AGENT.recommend(existing, top_k=3)
print("Intent:", result_t1['query'])
for r in result_t1['recommendations']:
    print(f"  #{r['rank']} {r.get('title','')[:60]}")

time.sleep(3)

print("\nTurn 2: user adds 'I want something specifically for dry hair'")
result_t2 = REC_AGENT.recommend(
    existing,
    conversation_context="User said: I want something specifically for dry hair",
    top_k=3
)
print("Intent:", result_t2['query'])
for r in result_t2['recommendations']:
    print(f"  #{r['rank']} {r.get('title','')[:60]}")

with open("/content/work/coldstart_multiturn.json", "w") as f:
    json.dump({"cold_start": result_cold, "turn_1": result_t1, "turn_2": result_t2}, f, indent=2, default=str)
print("\nsaved -> /content/work/coldstart_multiturn.json")

=== COLD-START DEMO (no prior reviews) ===
Inferred intent: This user is most likely to want a mid-range, versatile, and moderately-priced product, such as a 40-inch 4K smart TV with average-to-good reviews, a balance of features and price, and a brand reputation for reliability, reflecting their average rating of 3.5.
  #1 INFITRANS Hollywood Style LED Vanity Mirror with Dimmable Li
     why: The INFITRANS Hollywood Style LED Vanity Mirror has a high average rating of 4.5* and is likely to d
  #2 50pcs Clear Nail Art Fan-shaped Color Card, False Fake Nail 
     why: The 50pcs Clear Nail Art Fan-shaped Color Card has a perfect average rating of 5.0* and is likely to
  #3 Phonak TVLink II Base Station
     why: The Phonak TVLink II Base Station has a high average rating of 4.2* and may appeal to the user's pot
  #4 Deadpool Steelbook 2 Year Anniversary Edition 4K Ultra HD Bl
     why: The Deadpool Steelbook 2 Year Anniversary Edition 4K Ultra HD Blu-ray has a high average rating of 4
  